ライブラリ

In [7]:
!pip install mediapipe==0.10.20 opencv-python scikit-learn numpy pandas


画像1枚からランドマーク抽出

In [8]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)
mp_drawing = mp.solutions.drawing_utils

def extract_landmarks_from_image(path):
    image = cv2.imread(path)
    if image is None:
        print("画像が読めません:", path)
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    if not results.multi_hand_landmarks:
        print("手が検出できません:", path)
        return None

    # 座標を1次元に並べる（21点 × (x,y,z) = 63次元）
    lm = results.multi_hand_landmarks[0]
    data = []
    for p in lm.landmark:
        data.extend([p.x, p.y, p.z])

    return data


フォルダ内の画像を一気に処理（エラー対応付き）

In [9]:
import glob

def extract_folder(folder, label):
    features = []
    paths = sorted(glob.glob(folder + "/*.jpg") +glob.glob(folder + "/*.JPG"))

    for p in paths:
        f = extract_landmarks_from_image(p)
        if f is not None:
            features.append([label] + f)

    print(f"{folder} → 抽出できた枚数: {len(features)}")
    return features


あ行をCSVに保存

In [10]:
import pandas as pd

data = []

data += extract_folder("/content/drive/MyDrive/sign_letter_ai/data_a ", "a")
data += extract_folder("/content/drive/MyDrive/sign_letter_ai/data_i ", "i")
data += extract_folder("/content/drive/MyDrive/sign_letter_ai/data_u ", "u")
data += extract_folder("/content/drive/MyDrive/sign_letter_ai/data_e ", "e")
data += extract_folder("/content/drive/MyDrive/sign_letter_ai/data_o ", "o")

df = pd.DataFrame(data)
df.to_csv("/content/drive/MyDrive/sign_letter_ai/landmarks.csv", index=False)

print("保存完了！")

/content/drive/MyDrive/sign_letter_ai/data_a  → 抽出できた枚数: 26
/content/drive/MyDrive/sign_letter_ai/data_i  → 抽出できた枚数: 34
/content/drive/MyDrive/sign_letter_ai/data_u  → 抽出できた枚数: 32
/content/drive/MyDrive/sign_letter_ai/data_e  → 抽出できた枚数: 32
/content/drive/MyDrive/sign_letter_ai/data_o  → 抽出できた枚数: 33
保存完了！


学習モデルを作る（LightGBM or RandomForest）

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("/content/drive/MyDrive/sign_letter_ai/landmarks.csv")

X = df.iloc[:, 1:]  # 座標
y = df.iloc[:, 0]   # ラベル

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=500)
model.fit(X_train, y_train)

print("精度:", model.score(X_test, y_test))


精度: 0.84375


モデル保存

In [12]:
import pickle

with open("/content/drive/MyDrive/sign_letter_ai/ai_model.pkl", "wb") as f:
    pickle.dump(model, f)


1枚の画像で推論テスト

In [ ]:
test = extract_landmarks_from_image("/content/drive/MyDrive/sign_letter_ai/data_o /o_02.JPG")

if test is None:
    print("画像から座標が取れませんでした（ファイル名・手の向き・画像の明るさを確認）")
else:
    pred = model.predict([test])
    print("判定:", pred[0])




判定: o


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
